# Project 25: Water Usage & Conservation Analysis

**Industry:** Environment / Public Utilities  
**Business scenario:** Analyze household or building water-meter readings to understand consumption patterns, identify high usage, and suggest conservation measures.

This notebook completes all 15 tasks from the project brief using `25_water_consumption.csv`.

**How to use in Google Colab**
1. Open this notebook in Google Colab.
2. Run the cells from top to bottom.
3. If the CSV is not found automatically, the first data-loading cell will ask you to upload it.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the CSV and inspect dimensions, columns, and data types

In [ ]:
# The notebook first looks for the CSV in the current folder.
# In Google Colab, upload the CSV when prompted if it is not already present.
file_name = "25_water_consumption.csv"

if not os.path.exists(file_name):
    try:
        from google.colab import files
        print("Please upload the file: 25_water_consumption.csv")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
    except ImportError:
        file_name = input("Enter the CSV file path: ").strip()

df = pd.read_csv(file_name)

print("Dataset shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst five rows:")
display(df.head())

## 3. Identify missing values and duplicate Meter_Record_ID entries

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing_Count"))

duplicate_mask = df["Meter_Record_ID"].duplicated(keep=False)
print("Rows involved in duplicate Meter_Record_ID values:", int(duplicate_mask.sum()))
print("Number of extra duplicate IDs:", int(df["Meter_Record_ID"].duplicated().sum()))

print("\nDuplicate meter records:")
display(df.loc[duplicate_mask].sort_values("Meter_Record_ID"))

# Remove repeated meter-record IDs, keeping the first occurrence.
before = len(df)
df = df.drop_duplicates(subset="Meter_Record_ID", keep="first").copy()
print(f"Removed {before - len(df)} duplicate rows. Remaining rows: {len(df)}")

## 4. Handle missing Water_Usage_Liters values

The missing values are filled using the **median water usage for the same Household_Type**. If a household type has no available median, the overall median is used. Median is selected because it is less affected by unusually high or low readings than the mean.

In [ ]:
missing_before = df["Water_Usage_Liters"].isna().sum()
overall_median = df["Water_Usage_Liters"].median()

df["Water_Usage_Liters"] = df.groupby("Household_Type")["Water_Usage_Liters"].transform(
    lambda values: values.fillna(values.median())
)
df["Water_Usage_Liters"] = df["Water_Usage_Liters"].fillna(overall_median)

print("Missing usage values before filling:", missing_before)
print("Missing usage values after filling:", df["Water_Usage_Liters"].isna().sum())

## 5. Convert Reading_Date to datetime and extract month and weekday

In [ ]:
df["Reading_Date"] = pd.to_datetime(df["Reading_Date"], errors="coerce")
print("Rows with invalid dates:", df["Reading_Date"].isna().sum())

df = df.dropna(subset=["Reading_Date"]).copy()
df["Month"] = df["Reading_Date"].dt.to_period("M").astype(str)
df["Weekday"] = df["Reading_Date"].dt.day_name()

display(df[["Reading_Date", "Month", "Weekday"]].head())

## 6. Calculate total water consumption and average consumption per household

In [ ]:
household_summary = (
    df.groupby("Household_ID")["Water_Usage_Liters"]
      .agg(Total_Consumption_Liters="sum",
           Average_Consumption_Liters="mean",
           Number_of_Readings="count")
      .sort_values("Total_Consumption_Liters", ascending=False)
)

print("Household consumption summary:")
display(household_summary.head(10))

print("Overall total consumption (liters):", round(df["Water_Usage_Liters"].sum(), 2))
print("Average consumption per household (liters):",
      round(household_summary["Total_Consumption_Liters"].mean(), 2))

## 7. Calculate month-to-month consumption change for each household

In [ ]:
monthly_household = (
    df.groupby(["Household_ID", "Month"])["Water_Usage_Liters"]
      .sum()
      .reset_index()
      .sort_values(["Household_ID", "Month"])
)

monthly_household["Change_Liters"] = (
    monthly_household.groupby("Household_ID")["Water_Usage_Liters"].diff()
)
monthly_household["Change_Percent"] = (
    monthly_household.groupby("Household_ID")["Water_Usage_Liters"].pct_change() * 100
)

display(monthly_household.head(15))

## 8. Rank household types by total water usage

In [ ]:
household_type_summary = (
    df.groupby("Household_Type")["Water_Usage_Liters"]
      .agg(Total_Usage_Liters="sum", Average_Usage_Liters="mean",
           Reading_Count="count")
      .sort_values("Total_Usage_Liters", ascending=False)
)
household_type_summary["Rank"] = range(1, len(household_type_summary) + 1)

display(household_type_summary)

## 9. Compare average consumption across building types and zones

In [ ]:
building_summary = (
    df.groupby("Building_Type")["Water_Usage_Liters"]
      .agg(Average_Usage_Liters="mean", Total_Usage_Liters="sum",
           Reading_Count="count")
      .sort_values("Average_Usage_Liters", ascending=False)
)

zone_summary = (
    df.groupby("Zone")["Water_Usage_Liters"]
      .agg(Average_Usage_Liters="mean", Total_Usage_Liters="sum",
           Reading_Count="count")
      .sort_values("Total_Usage_Liters", ascending=False)
)

print("Building-type comparison:")
display(building_summary)
print("Zone comparison:")
display(zone_summary)

## 10. Identify the 10 households with the highest total water consumption

In [ ]:
top_10_households = household_summary.head(10)
display(top_10_households)

top_10_households["Total_Consumption_Liters"].sort_values().plot(
    kind="barh", figsize=(9, 5), title="Top 10 Households by Total Water Consumption"
)
plt.xlabel("Total consumption (liters)")
plt.ylabel("Household ID")
plt.tight_layout()
plt.show()

## 11. Find the month with the highest overall water usage

In [ ]:
monthly_total = (
    df.groupby("Month")["Water_Usage_Liters"]
      .sum()
      .sort_index()
)

highest_month = monthly_total.idxmax()
highest_month_usage = monthly_total.max()

print("Month with highest total water usage:", highest_month)
print("Total usage in that month (liters):", round(highest_month_usage, 2))

monthly_total.plot(kind="line", marker="o", figsize=(11, 5),
                   title="Monthly Water Consumption Trend")
plt.xlabel("Month")
plt.ylabel("Total consumption (liters)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Use a dictionary for zone-wise totals and a set for unique zones

In [ ]:
zone_totals = df.groupby("Zone")["Water_Usage_Liters"].sum().to_dict()
unique_zones = set(df["Zone"].dropna().unique())

print("Zone-wise consumption dictionary:")
print(zone_totals)
print("\nUnique zones:")
print(unique_zones)

## 13. Create a function to classify consumption as Low, Normal, or High

For this project, the following clear thresholds are used for an individual meter reading:
- **Low:** below 4,000 liters
- **Normal:** from 4,000 through 8,000 liters
- **High:** above 8,000 liters

These are simple project-analysis thresholds, not official water-use standards. They can be adjusted to suit a real utility's billing period and local benchmarks.

In [ ]:
def classify_consumption(liters):
    if pd.isna(liters):
        return "Unknown"
    elif liters < 4000:
        return "Low"
    elif liters <= 8000:
        return "Normal"
    else:
        return "High"

df["Consumption_Category"] = df["Water_Usage_Liters"].apply(classify_consumption)
display(df[["Household_ID", "Water_Usage_Liters", "Consumption_Category"]].head(10))

print("Number of records in each category:")
display(df["Consumption_Category"].value_counts().to_frame("Record_Count"))

## 14. Use NumPy for mean, median, variance, and standard deviation

In [ ]:
usage_array = df["Water_Usage_Liters"].to_numpy()

print("Mean (liters):", round(np.mean(usage_array), 2))
print("Median (liters):", round(np.median(usage_array), 2))
print("Variance (liters squared):", round(np.var(usage_array), 2))
print("Standard deviation (liters):", round(np.std(usage_array), 2))

## 15. Detect unusually high readings using the IQR method and compare household types

In [ ]:
Q1 = df["Water_Usage_Liters"].quantile(0.25)
Q3 = df["Water_Usage_Liters"].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

outliers = df[
    (df["Water_Usage_Liters"] < lower_limit) |
    (df["Water_Usage_Liters"] > upper_limit)
].copy()

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower IQR limit:", round(lower_limit, 2))
print("Upper IQR limit:", round(upper_limit, 2))
print("Number of unusual records:", len(outliers))

print("\nUnusually high readings:")
display(outliers.sort_values("Water_Usage_Liters", ascending=False).head(20))

print("\nAverage usage by household type:")
average_by_household_type = (
    df.groupby("Household_Type")["Water_Usage_Liters"]
      .mean()
      .sort_values(ascending=False)
      .to_frame("Average_Usage_Liters")
)
display(average_by_household_type)

average_by_household_type.plot(kind="bar", legend=False, figsize=(8, 5),
                               title="Average Water Usage by Household Type")
plt.ylabel("Average consumption (liters)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 16. Create a simple WaterMeter class

In [ ]:
class WaterMeter:
    def __init__(self, meter_id, household_type):
        self.meter_id = meter_id
        self.household_type = household_type

    def meter_details(self):
        return {
            "Meter_ID": self.meter_id,
            "Household_Type": self.household_type
        }

# Example object using a record from the dataset
first_meter = WaterMeter(
    df.iloc[0]["Meter_Record_ID"],
    df.iloc[0]["Household_Type"]
)
print(first_meter.meter_details())

## 17. Five water-conservation recommendations based on the analysis

In [ ]:
# Generate recommendations using the observed dataset patterns.
highest_household_type = average_by_household_type.index[0]
highest_building_type = building_summary["Average_Usage_Liters"].idxmax()
highest_zone = zone_summary["Total_Usage_Liters"].idxmax()
high_category_count = int((df["Consumption_Category"] == "High").sum())
outlier_count = len(outliers)

recommendations = [
    f"Prioritize water audits and awareness campaigns for {highest_household_type} households, "
    "which have the highest average recorded usage.",
    f"Inspect plumbing, cooling, and shared facilities in {highest_building_type} buildings "
    "because this building type has the highest average recorded usage.",
    f"Conduct a zone-level leak detection and meter-check campaign in {highest_zone}, "
    "the zone with the highest total recorded consumption.",
    f"Review the {high_category_count} readings classified as High and contact the relevant "
    "households to check for leaks, wastage, or unusual activity.",
    f"Investigate the {outlier_count} IQR-flagged unusual readings, install low-flow fixtures "
    "where suitable, and promote rainwater harvesting and reuse of greywater."
]

for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

## 18. Final deliverables / project summary

In [ ]:
print("ZONE-WISE CONSUMPTION TABLE")
display(zone_summary)

print("HOUSEHOLD TYPE COMPARISON")
display(household_type_summary)

print("MONTHLY WATER-USE TREND DATA")
display(monthly_total.to_frame("Total_Usage_Liters"))

print("TOP 10 HIGH-CONSUMPTION HOUSEHOLDS")
display(top_10_households)

print("STATISTICAL SUMMARY")
statistics_summary = pd.Series({
    "Mean": np.mean(usage_array),
    "Median": np.median(usage_array),
    "Variance": np.var(usage_array),
    "Standard Deviation": np.std(usage_array),
    "IQR Upper Limit": upper_limit
})
display(statistics_summary.to_frame("Value"))

print("FIVE CONSERVATION RECOMMENDATIONS")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

## Conclusion

The analysis cleans the water-meter dataset, summarizes consumption by household, building type, zone, and month, identifies high-use households and unusual readings, and produces practical conservation recommendations. Interpret the results in the context of the dataset's reading period and the number of readings per household.